To predict the probability of frog presence ($p$) for a specific site, we use the weights calculated by our Gradient Ascent algorithm. This process happens in two stages: calculating the Log-Odds ($z$) and passing them through the Sigmoid Function.

$$\Large p = \frac{1}{1 + e^{-(-117.5187 - 0.0007(\text{Distance}) + 6.9139(\text{Temp Mid}) - 0.6628(\text{Temp Mid} - \text{Mean})^2 + 0.0388(\text{Altitude}))}}$$


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import ipywidgets as widgets
from IPython.display import display

In [2]:
display(widgets.HTML("<h3>🐸 Interactive Frog Habitat Predictor</h3><p>Adjust the environmental parameters below to calculate the probability of frog presence in real-time.</p>"))

HTML(value='<h3>🐸 Interactive Frog Habitat Predictor</h3><p>Adjust the environmental parameters below to calcu…

In [2]:
# 1. LOAD DATA GLOBALLY VIA GITHUB RAW URL (Fixes all Binder path issues)
csv_url = "https://raw.githubusercontent.com/Rent45/Frog-Data-Analysis/main/Data/frogs.csv"
df = pd.read_csv(csv_url, index_col=0)

In [3]:
# 2. FEATURE ENGINEERING
df["temp_mid"] = (df["meanmax"] + df["meanmin"]) / 2
temp_mean = df['temp_mid'].mean()
df['temp_mid_sq'] = (df['temp_mid'] - temp_mean) ** 2

X_df = df[['distance', 'temp_mid', 'temp_mid_sq', 'altitude']]
X_df_with_intercept = sm.add_constant(X_df)
X = X_df_with_intercept.to_numpy()
Y = df['pres.abs'].to_numpy()

In [4]:
# 3. SCALE DATA
X_scaled = X.copy()
for i in range(1, X.shape[1]):
    X_scaled[:, i] = (X[:, i] - np.mean(X[:, i])) / np.std(X[:, i])

In [5]:
# 4. TRAINING LOOP
def gradient(x, y, rate, n):
    weigh = np.zeros(x.shape[1]) # Automatically sized to 5
    m = len(y)
    for _ in range(n):
       z = np.dot(x, weigh)
       p = 1 / (1 + np.exp(-z))
       grad = np.dot(x.T, (y - p)) / m
       weigh += rate * grad
    return weigh
best_weights = gradient(X_scaled, Y, rate=2.0, n=20000)

In [6]:
# 5. UNSCALE WEIGHTS & CALCULATE COVARIANCE
means = np.mean(X[:, 1:], axis=0)
stds = np.std(X[:, 1:], axis=0)
scaled_intercept = best_weights[0]
scaled_features = best_weights[1:]

unscaled_features = scaled_features / stds
unscaled_intercept = scaled_intercept - np.sum((scaled_features * means) / stds)
unscaled_weights = np.insert(unscaled_features, 0, unscaled_intercept)

p_train = 1 / (1 + np.exp(-np.dot(X, unscaled_weights)))
W = np.diag(p_train * (1 - p_train))
fisher_info = np.dot(np.dot(X.T, W), X)
cov_matrix = np.linalg.inv(fisher_info)
standard_errors = np.sqrt(np.diag(cov_matrix))

In [7]:
# 6. UI SUMMARY FUNCTION
def predict_and_summarize(x_new, weights, cov_mat, std_errs):
    z_90, z_95, z_99 = 1.645, 1.960, 2.576
    z_val = np.dot(x_new, weights)
    se_z = np.sqrt(np.dot(np.dot(x_new, cov_mat), x_new))

    def sigmoid(val): return 1 / (1 + np.exp(-np.clip(val, -500, 500)))
    def format_prob(val): return f"{val:>8.2e}%" if val < 0.001 else f"{val:>8.2f}%"

    p_est = sigmoid(z_val) * 100
    p_low_90, p_high_90 = sigmoid(z_val - z_90 * se_z) * 100, sigmoid(z_val + z_90 * se_z) * 100
    p_low_95, p_high_95 = sigmoid(z_val - z_95 * se_z) * 100, sigmoid(z_val + z_95 * se_z) * 100
    p_low_99, p_high_99 = sigmoid(z_val - z_99 * se_z) * 100, sigmoid(z_val + z_99 * se_z) * 100

    # Format the probability strings
    p_est_str = format_prob(p_est)
    p_90_str = f"[{format_prob(p_low_90)}, {format_prob(p_high_90)}]"
    p_95_str = f"[{format_prob(p_low_95)}, {format_prob(p_high_95)}]"
    p_99_str = f"[{format_prob(p_low_99)}, {format_prob(p_high_99)}]"

    # Expanded table width to 108 to comfortably fit scientific notation
    print("-" * 108)
    print(f"{'Feature':<12} | {'Estimate':>10} | {'90% CI':>24} | {'95% CI':>24} | {'99% CI':>24}")
    print("-" * 108)

    # Restored all probability intervals
    print(f"{'Probability':<12} | {p_est_str:>10} | {p_90_str:>24} | {p_95_str:>24} | {p_99_str:>24}")
    print("-" * 108)

    feature_names = ["Intercept", "Distance", "Temp_Mid", "Temp_Mid_Sq", "Altitude"]
    for i, name in enumerate(feature_names):
        w, se = weights[i], std_errs[i]

        ci90 = f"[{w-(z_90*se):>9.4f}, {w+(z_90*se):>8.4f}]"
        ci95 = f"[{w-(z_95*se):>9.4f}, {w+(z_95*se):>8.4f}]"
        ci99 = f"[{w-(z_99*se):>9.4f}, {w+(z_99*se):>8.4f}]"

        print(f"{name:<12} | {w:>10.4f} | {ci90:>24} | {ci95:>24} | {ci99:>24}")
    print("-" * 108)

In [2]:
# 7. INTERACTIVE WIDGET
def interactive_habitat_predictor(distance, temperature, altitude):
    centered_temp_sq = (temperature - temp_mean) ** 2
    new_site_binder = np.array([1.0, distance, temperature, centered_temp_sq, altitude])
    predict_and_summarize(new_site_binder, unscaled_weights, cov_matrix, standard_errors)

display(widgets.HTML("<h3>🐸 Interactive Frog Habitat Predictor</h3><p>Adjust the environmental parameters below to calculate the probability of frog presence in real-time.</p>"))

widgets.interact(
    interactive_habitat_predictor,
    distance=widgets.FloatSlider(value=500, min=0, max=5000, step=50, description='Distance (m):', layout=widgets.Layout(width='500px')),
    temperature=widgets.FloatSlider(value=15.0, min=-5, max=40, step=0.5, description='Temp (°C):', layout=widgets.Layout(width='500px')),
    altitude=widgets.FloatSlider(value=1200, min=0, max=4000, step=50, description='Altitude (m):', layout=widgets.Layout(width='500px'))
);

HTML(value='<h3>🐸 Interactive Frog Habitat Predictor</h3><p>Adjust the environmental parameters below to calcu…

interactive(children=(FloatSlider(value=500.0, description='Distance (m):', layout=Layout(width='500px'), max=…